# 21.21 / MP-09. USD/Omniverse: геометрический preflight

**Статус: технический экспорт 12 наблюдённых auto-фаз Ника в USD выполнен; просмотр в Omniverse и независимая анатомическая проверка не выполнены.** Вопрос: помогает ли проигрывание фаз найти ошибки единиц, ориентации, координат, топологии и движения, которые не заметны в статическом QC? Положительный результат будущего визуального опыта — воспроизводимый перечень конкретных дефектов с привязкой к фазам и подтверждением на исходной КТ. Отрицательный результат также допустим. Просмотр не повышает доказательный статус сегментации, кинематики или УО/ФВ.

Первичные сведения: [MONAI Physio Tutorial 5](https://project-monai.github.io/monai-physio/tutorials.html#tutorial-5-vtk-surfaces-to-animated-usd), [WorkflowConvertVTKToUSD](https://project-monai.github.io/monai-physio/api/workflows.html#vtk-to-usd), [OpenUSD: линейные единицы сцены](https://openusd.org/25.11/api/group___usd_geom_linear_units__group.html).

## Основной MP-09 preflight на Нике

[Read-only повтор](mp08_09_nix_preflight_results.json) прочитал существующий STL фазы 00 Ника: `307 036` треугольников, bounds внутри FOV КТ. Это грубая проверка координат, а не доказательство полной анатомии: маска достигает верхней границы FOV. Поля MP-03 построены для остальных 11 фаз на исходной сетке, но не допущены по независимой анатомии. [Технический экспорт USD](mp09_nix_usd_report.md) содержит отдельно сегментированные auto-поверхности всех 12 фаз; это визуальная последовательность, а не перенос принятой поверхности через поле. Синтетический USD-smoke ниже остаётся тестом единиц и API. Старый preflight Георга сохранён как исторический.


## Прочитанные входы и результат

**Факт о файлах, 28.09.2026.** Read-only скрипт `mp08_09_preflight.py` проверил существующую производную STL-поверхность целого сердца фазы 00 Georg: 719662 треугольника; bounds лежат внутри FOV исходной NIfTI на её affine-сетке. Это лишь грубая проверка совместимости координат. Не проверены анатомическая правильность, замкнутость, нормали, совпадение с маской, соответствие вершин между фазами и движение. Заголовок NIfTI не содержит единиц; конверсионный манифест указывает RAS+ и миллиметры.

**Окружения.** Первоначальный preflight выполнялся в системном Python без `monai_physio`, `itk`, `pxr`; последующий синтетический smoke — в установленном проектном `.venv` с MONAI Physio 2026.09.1, ITK и OpenUSD. Omniverse не запускался. Экспорт реальной поверхности в Git не выполнялся.

## Синтетический VTK→USD smoke (28.09.2026)

**Модельный результат.** `WorkflowConvertVTKToUSD` экспортировал три положения маленькой сферы в `mp08_09_synthetic_motion.usda`. После открытия файла через OpenUSD получены один Mesh, отсчёты 0, 1, 2 и по 74 вершины в каждом. Исходные смещения VTK в миллиметрах: (0,0,0), (2,0,0), (2,1,1.5). Записанные и повторно считанные смещения центроидов USD в **метрах**: (0,0,0), (0.002,0,0), (0.002,0.0015,−0.001). Они совпали с ожидаемым преобразованием `(x,y,z) мм → (x,z,−y) м` в пределах 5×10⁻⁶ м. Readback проверяется assert в скрипте; предыдущая версия JSON ошибочно называла эти значения миллиметрами и заменена.

Сцена имеет `metersPerUnit = 1`, `upAxis = Y`, `timeCodesPerSecond = 2`. `framesPerSecond` возвращает значение по умолчанию 24 и не является измеренной частотой сердечного цикла. Эти метаданные оставлены как записал экспортёр: изменение одного лишь `metersPerUnit` на 0.001 при уже переведённых в метры точках дало бы неверный масштаб. Подтверждены сериализация и численный readback синтетической сцены; рендер в Omniverse и анатомическое соответствие не проверялись. Числа сохранены в `mp08_09_synthetic_results.json`.

## Контракт будущего экспорта и QC

Текущий локальный файл `runs/mp09_nix_observed_auto_phases.usdc` содержит 12 отдельно сегментированных поверхностей; его повторное чтение подтвердило координаты, размерность и сохранность фаз. Топология меняется между фазами, поэтому интерполяция промежуточных форм не валидирована, а порядковый индекс не означает измеренное время ЭКГ. После принятия геометрии и поля MP-03/04 подготовить вне Git согласованные фазовые поверхности в общей RAS-системе, с зафиксированным масштабом, ориентацией и происхождением. Проверять **вместе** фактические координаты точек USD, `metersPerUnit`, `upAxis` и ориентацию; сверить с анатомическими ориентирами КТ, а не только с bounding box.

Для анимации одной согласованной сетки проверить постоянство числа и порядка вершин. В текущем экспорте меняются и вершины, и топология; результат читать только в дискретных фазах. Провести визуальный обзор всех фаз совместно с исходными CT и масками: отражение, сдвиг, масштаб, выход за FOV, самопересечения, скачки, скрытие объектов. Сохранить журнал дефектов и версии конвертера. Поля регистрации существуют, но не прошли независимую анатомическую проверку; временного QC также нет. Поэтому готовность *валидированной* пациентской анимации остаётся `false`.